# 00 - Por que aparece MCP: el problema ad hoc

## Que vas a aprender

Vas a ver por que una integracion directa entre un LLM y una API externa puede servir para una demo, pero se vuelve dificil de mantener cuando varias apps necesitan la misma capacidad.

En esta clase usamos un ejemplo conductor: **un MCP local que permite crear repos privados y escribir archivos en GitHub**.


## Teoria de la lecture

Una integracion ad hoc es una solucion puntual: una funcion, wrapper o script que conecta un caso especifico con una API. El problema no es que este mal para empezar; el problema aparece cuando se multiplica.

| Situacion | Que pasa al principio | Que problema aparece despues |
|---|---|---|
| Un bot crea repos | Una funcion alcanza | El schema queda pegado al bot |
| Un IDE crea repos | Otro wrapper parece rapido | Los nombres de campos cambian |
| Un host con LLM crea repos | Se arma otra funcion | No hay permisos ni auditoria comunes |

MCP propone publicar una capacidad comun con contrato, discovery, seguridad y auditoria.


## Diagrama textual

```text
Antes de MCP

Bot        -> wrapper_a(repo_name, text)       -> GitHub API
IDE        -> wrapper_b(name, description)     -> GitHub API
LLM Host   -> wrapper_c(project, goal)         -> GitHub API

Problema: tres contratos para la misma capacidad.
```

```text
Con MCP

Bot / IDE / LLM Host -> MCP Client -> MCP Server GitHub -> GitHub API

Resultado: una capacidad gobernada y reusable.
```


## Paso que construimos del MCP GitHub

En este primer paso todavia no usamos el SDK. Solo identificamos el problema: queremos una unica capacidad para crear repositorios y README, sin duplicar wrappers.


In [ ]:
def cli_create_repo(repo_name, readme_text):
    return {"source": "cli", "repo_name": repo_name, "readme_text": readme_text}


def web_new_repository(name, description, initialize):
    return {"source": "web", "name": name, "description": description, "initialize": initialize}


def agent_bootstrap_project(project, goal):
    return {"source": "agent", "project": project, "goal": goal}

examples = [
    cli_create_repo("aem4-demo", "README inicial"),
    web_new_repository("aem4-demo", "Demo MCP", True),
    agent_bootstrap_project("aem4-demo", "mostrar MCP"),
]

for item in examples:
    print(item)


## Interpretacion

Los tres ejemplos intentan hacer algo parecido, pero cada uno inventa su propio contrato. Si mañana agregamos permisos, branch, auditoria o validacion, habria que corregir tres lugares.

## Practica

Escribi una propuesta de contrato unico para crear un repo privado. Debe incluir:

| Campo | Tipo | Obligatorio | Motivo |
|---|---|---|---|
| `name` | string | si | nombre del repo |
| `description` | string | no | explica el objetivo |
| `private` | boolean | no | seguridad por defecto |
